# **Grad-CAM: Практика**

Добро пожаловать на урок по методу GradCAM в рамках нашего [курса](https://stepik.org/a/198640) по объяснимому искусственному интеллекту. На этой практической части будем работать с GradCAM.

GradCAM (Gradient-weighted Class Activation Mapping) – один из наиболее популярных методов визуализации активаций. Популярность обусловлена количество модификаций метода и простотой (ведь это просто надстройка над картами активации).

Как и карты активации, метод позволяет понять, какие именно части изображения наиболее сильно повлияли на *итоговую классификацию*.

В этом уроке вы рассмотрите и повторите, как работает GradCAM.

Приятного кодинга!

<img src="https://raw.githubusercontent.com/SadSabrina/open-xai-materials/main/assets/pawel-czerwinski-CEr4ljp-MSh4-unsplash.jpg" alt="pawel-czerwinski-CEr4ljp-MSh4-unsplash" border="0">

**Шаг 1. Импортируем необходимые библиотеки.**

In [ ]:
!pip install grad-cam -q

In [ ]:
import os
import cv2
import numpy as np
import requests
from io import BytesIO
from matplotlib import pyplot as plt
from matplotlib.pyplot import imshow
from PIL import Image
import torch
from torch import nn
from torchvision import models, transforms
from torch.nn import functional as F
from torch import nn as nn
from torch.autograd import Variable
from torchvision.models import resnet50, ResNet50_Weights, densenet201, DenseNet201_Weights


#Для работы с реализацией GradCAM под капотом
import warnings
warnings.filterwarnings('ignore')
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
from pytorch_grad_cam.utils.image import show_cam_on_image, deprocess_image, preprocess_image


**Вспоминание теории.**

Вспомним, что нам нужно, чтобы построить GradCAM.

- Получить Class Activation Map
- Взвесить полученную карту активации классов по градиенту

Благодаря гибкости pyTorch, эти действия Grad-CAM можно реализовать тремя путями:
- с использованием hooks, с которыми вы познакомились на практике по Guided backpropagation;
- с переопределением последних слоёв сети;
- используя уже реализованный метод под капотом.

**В чем разница и что использовать?**

Как мы с вами прошли в теоретической части, для получения CAM необходим доступ к последнему сверточному слою модели. В свою очередь, реализации моделей могут являться разными, так что процесс получения карты признаков с последнего сверточного слоя необходимо оптимизировать под конкретную (в том числе вашу собственную) архитектуру. Поэтому удобнее и быстрее использовать [готовые методы](https://pypi.org/project/grad-cam/).

Но реализация получения выходных значений методов интерпретации вручную, полезна для понимания методов и их недостатков "изнутри". Поэтому в этой практике вы как поработаете с библиотекой, так и реализуете GradCAM самостоятельно на примере двух архитектур `ResNet` и `DenseNet`.

Загрузим модели, с которыми будем работать.

In [ ]:
densenet = densenet201(weights=DenseNet201_Weights.IMAGENET1K_V1)
resnet = resnet50(weights=ResNet50_Weights.IMAGENET1K_V1)

Выполните вызов каждой переменной поочереди и внимательно посмотрите на архитектуры моделей.

In [ ]:
#densenet

In [ ]:
#resnet

В прошлом уроке, когда вы строили Class Activation Map для ResNet, вы задействовали три сосотавляющие архитектуры:

- Карты активации с последнего сверточного слоя
- Глобальное усреднение над картами активации
- Линейный слой, прогнозирующий вероятность

Для построения карт для *других* моделей, также нужен доступ к эти компонентам архитектуры.

На верхнем уровне, каждую нейронную сеть можно рассматривать как последовательность модулей, выполняющих операции над входным объектом. Доступ к модулям сети можно молучить, вызвав итератор `model.children()`. Однако, чтобы добраться до последнего сверточного слоя недостаточно получить доступ к модулю. Необходимо пробираться детальнее по модели.

Для этого, как уже было описано выше, можно либо немного переопределять архитектуру, либо прикрепить `hook` к нужному слою, либо пользоваться готовыми решениями.

Для понимания, последовательно рассмотрим каждый метод.

Прежде напишем вспомогательные функции и обработаем изображение для подачи его на вход нейронной сети.

In [ ]:
# Загрузка изображения
url = 'https://github.com/SadSabrina/explainable_AI_course/blob/main/HW_module10.1_gradient_methods/cat_and_dog.jpg?raw=true'

image_bytes = requests.get(url).content
image = Image.open(BytesIO(image_bytes)) # Снова рассмотрим конкретный пример x_0

plt.figure(figsize=(8,10))
plt.axis('off')
plt.imshow(image);

In [ ]:
#Предобработка

preprocess = transforms.Compose([
   transforms.Resize((224,224)),
   transforms.ToTensor(),
   transforms.Normalize(
   mean=[0.485, 0.456, 0.406],
   std=[0.229, 0.224, 0.225]
)
])

# Функция для простого ресайз'а изображения
display = transforms.Compose([
    transforms.Resize((224,224))
    ])

tensor = preprocess(image)
pred = Variable((tensor.unsqueeze(0)), requires_grad=True)

**Способ 1. Доопределение модели.**

Доопределение модели реализуется при помощи написания собственного класса сети. При написании новой модели необходимо выделить две части:

- ту, что извлекает признаки;
- ту, что осуществляет по полученным признакам классификацию.  

Их запишем в атрибуты класса `features` и `classifier` соответственно.


In [ ]:
#ResNet

class ModifiedResNet(nn.Module):
    def __init__(self):

        """
        Constructor of the ModifiedResNet class
        """

        super(ModifiedResNet, self).__init__()

        self.resnet = resnet50(weights=ResNet50_Weights.IMAGENET1K_V1) # в атрибуте resnet сохраним модель-оригинал

        self.features = nn.Sequential(*list(self.resnet.children())[:-2]) # в атрибуте features сохраним часть, извлекающую признаки

        self.classifier = nn.Sequential(*list([nn.AdaptiveAvgPool2d((1, 1))] + [nn.Flatten()] +[self.resnet.fc])) # в атрибуте classifier релализуем последовательсть из пулинга и извлечения итогового прогноза

    def forward(self, x: torch.Tensor) -> torch.Tensor:

        """
        Forward step of the model
        """

        x = self.features(x)
        x = self.classifier(x)
        return x

resnet = ModifiedResNet()
original_resnet = resnet50(weights=ResNet50_Weights.IMAGENET1K_V1)

f_ex = resnet.features
classification = resnet.classifier

resnet.eval()
original_resnet.eval();

**Quiz 1 Получите прогноз от модифицированной модели. Какой номер класса прогнозирует resnet?**

In [ ]:
print('Прогноз модели-оригинала: ', # Ваш код здесь)
print('Прогноз модифицированной модели: ', # Ваш код здесь)

Получение признаков реализуем применив к inputу часть, извлекающую признаки, а класификационные оценки — применив к полученным признаками классификатор. Также запишем размеры полученной карты, поскольку она нам еще понадобится.

In [ ]:
f_map = f_ex(pred) # Получение карты признаков
_, N, H, W = f_map.size() # Извлечение размеров полученной карты

c_score = classification(f_map)[0, 179] # Получение оценки классификации

Вспомним формулу для построения GradCAM:

$$L^c_{\text{Grad-CAM}} = \mathrm{ReLU}\Big(\sum_k \alpha^c_k A^k\Big)$$

где $a^C_K$ — это усреднение по градиентам, полученным при backward pass:

$$\alpha^c_k = \frac{1}{Z}\sum_i\sum_j \frac{\partial y^c}{\partial A^k_{ij}}$$

Как получить градиенты backwarda? Один из способов — использовать `torch.autograd.grad()`.

В PyTorch `torch.autograd.grad()` используется для вычисления градиентов одних тензоров относительно других тензоров — это то, что нам нужно.

```
torch.autograd.grad(
    outputs,        # Тензор(ы), по которым нужно вычислить градиенты.
    inputs,         # Тензор(ы), относительно которых нужно вычислить градиенты.)
```


**Quiz 2 Вычислите градиент `f_map` относительно оценки `c_score`. Результат запишите в переменную grads. Сколько размерностей у выхода?**

In [ ]:
grads = torch.autograd.grad(c_score, f_map)

In [ ]:
# Ваш код здесь

**Quiz 3 Реализуйте вычисление весов как среднее по последним двум размерностям. Какой длины получается результат?**

Подсказка: взять среднее можно как `grads[0].mean()`

In [ ]:
w = # Ваш код здесь

In [ ]:
gradcam = torch.matmul(w, f_map.view(N, H*W))
gradcam = gradcam.view(H, W)

gradcam = nn.functional.relu(gradcam)

In [ ]:
gradcam_tensor = gradcam.unsqueeze(0).unsqueeze(0)  # Преобразовываем обратно в тензор

interpol = F.interpolate(gradcam_tensor, (224, 224), mode="bilinear") # Интерполируем получившуюся карту на нужный размер
interpol = interpol.squeeze(0).squeeze(0) #Подгатавливаем результат к визуализации
interpol = interpol.detach().numpy()

plt.imshow(display(image))
plt.imshow(interpol, alpha=0.5, cmap='jet')
plt.axis('off')
plt.show()

Если собирать всё воедино таким образом, можно написать следующую функцию

In [ ]:
def GradCAM_function(input_img, cl_sc: torch.Tensor | float, f_ex: nn.Module, classification: nn.Module) -> torch.Tensor:

    f_map = f_ex(input_img)
    _, N, H, W = f_map.size()

    c_score = classification(f_map)[0, cl_sc]

    grads = torch.autograd.grad(c_score, f_map) # считаем backward по оценке классификации относительно карты признаков
    w = grads[0][0].mean(-1).mean(-1)

    gradcam = torch.matmul(w, f_map.view(N, H*W)) # перемножаем карту на веса
    gradcam = gradcam.view(H, W)
    gradcam = nn.functional.relu(gradcam)

    return gradcam

In [ ]:
GradCAM_function(pred, 179, f_ex, classification)

И по определению контрфактической карты, немного поменя функцию выше можно построить Counterfactual GradCAM, которое может как дать, так и не дать дополнительную информацию.


**Quiz 4 Выберите на степик, как на основе функции для GradCAM построить Counterfactual GradCAM. На основе ответа доработайте функцию ниже. Получилось ли построить информативный пример для целевого класса?**

In [ ]:
def Counterfactual_GradCAM_function(input_img, cl_sc: torch.Tensor | float, f_ex: nn.Module, classification: nn.Module) -> torch.Tensor:

    f_map = f_ex(input_img)
    _, N, H, W = f_map.size()

    c_score = classification(f_map)[0, cl_sc]

    grads = torch.autograd.grad(c_score, f_map) # считаем backward по оценке классификации относительно карты признаков
    w = grads[0][0].mean(-1).mean(-1)

    gradcam = torch.matmul(w, f_map.view(N, H*W)) # ДООПРЕДЕЛИТЕ КОД
    gradcam = gradcam.view(H, W)
    gradcam = nn.functional.relu(gradcam)

    return gradcam


counterfactual_gc = Counterfactual_GradCAM_function(pred, 179, f_ex, classification).unsqueeze(0).unsqueeze(0)

interpol = F.interpolate(counterfactual_gc, (224, 224), mode="bilinear") # Интерполируем получившуюся карту на нужный размер
interpol = interpol.squeeze(0).squeeze(0) #Подгатавливаем результат к визуализации
interpol = interpol.detach().numpy()

plt.imshow(display(image))
plt.imshow(interpol, alpha=0.5, cmap='jet')
plt.axis('off')
plt.show()

**Quiz 5 Аналогично допишите класс для DenseNet. Совпадает ли прогноз модифицированной модели с оригинальной?**

In [ ]:
#DenseNet

class ModifiedDenseNet(nn.Module):
    def __init__(self):
        super(ModifiedDenseNet, self).__init__()
        self.densenet = models.densenet201(weights=DenseNet201_Weights.IMAGENET1K_V1)
        self.features = self.densenet.features
        self.classifier = nn.Sequential(*list([nn.AdaptiveAvgPool2d((1, 1))] + [nn.Flatten()] + [self.densenet.classifier]))

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

densenet = ModifiedDenseNet()

f_ex = densenet.features
classification = densenet.classifier

densenet.eval();

In [ ]:
print('Прогноз оригинальной модели: ', # Ваш код здесь)
print('Прогноз модифицированной модели: ', # Ваш код здесь )

In [ ]:
gradcam = GradCAM_function(pred, 254, f_ex, classification).unsqueeze(0).unsqueeze(0)
interpol = F.interpolate(gradcam, (224, 224), mode="bilinear") # Интерполируем карту на изображение
interpol = interpol.squeeze(0).squeeze(0).detach().numpy()


plt.axis('off')
plt.imshow(display(image))
plt.imshow(interpol, alpha=0.5, cmap='jet')
plt.show()

**Способ 2. Использование Hooks.**

Ещё один способ извлекать выходы forward и backward проходов — прикрепление хуков, с которыми вы познакомились ранее. Реализуем применение hooks для Densenet и ResNet.

In [ ]:
#DenseNet

# Словарь для хранения активаций и градиентов
activations = {}
gradients = {}

# Функция forward hook для сохранения активаций
def forward_hook(module, input, output):
    activations['conv_output'] = output

# Функция backward hook для сохранения градиентов
def backward_hook(module, grad_in, grad_out):
    gradients['conv_gradients'] = grad_out[0]


densenet = models.densenet201(weights=DenseNet201_Weights.IMAGENET1K_V1)

densenet.eval();

In [ ]:
forward_hook = densenet.features[-2].register_forward_hook(forward_hook)
backward_hook = densenet.features[-2].register_backward_hook(backward_hook)

prediction = densenet(pred)
cl_cls = prediction.argmax(dim=1)

densenet.zero_grad()

In [ ]:
prediction[0, cl_cls].backward()

In [ ]:
hook_f_map = activations['conv_output']
hook_weight = gradients['conv_gradients'].mean(axis=(2, 3)) # получаем веса, как среднее последних двух размерностей


_, N, H, W = hook_f_map.size()

gradcam = torch.matmul(hook_weight, hook_f_map.view(1920, 7*7)) # строим карту при помощи произведения

gradcam = gradcam.view(H, W).cpu().detach().numpy() # приводим карту к читаемому виду


gradcam = np.maximum(gradcam, 0) #ReLU


gradcam_tensor = torch.from_numpy(gradcam).float().unsqueeze(0).unsqueeze(0)  # Преобразовываем обратно в тензор
interpol = F.interpolate(gradcam_tensor, (224, 224), mode="bilinear").squeeze(0).squeeze(0) # Интерполируем карту на изображение


plt.axis('off')
plt.imshow(display(image))
plt.imshow(interpol, alpha=0.5, cmap='jet')
plt.show()

Аналогично, реализуйте взятие Hook для ResNet, выбрав необходимую часть(необходимый **слой**) сети. Совпала ли полученная карта с первой картой для модели?

In [ ]:
#ResNET

# Словарь для хранения активаций и градиентов
activations = {}
gradients = {}

# Функция forward hook для сохранения активаций
def forward_hook(module, input, output):
    activations['conv_output'] = output

# Функция backward hook для сохранения градиентов
def backward_hook(module, grad_in, grad_out):
    gradients['conv_gradients'] = grad_out[0]


resnet = resnet50(weights=ResNet50_Weights.IMAGENET1K_V1)

resnet.eval();

In [ ]:
forward_hook = resnet.layer4[-1].register_forward_hook(forward_hook)
backward_hook = resnet.layer4[-1].register_backward_hook(backward_hook)

prediction = resnet(pred) # получаем прогноз
cl_cls = prediction.argmax(dim=1) # извлекаем метку класса

resnet.zero_grad()

prediction[0, cl_cls].backward() # делаем backward pass

In [ ]:
# Аналогичное построение карты

hook_f_map = activations['conv_output']
hook_weight = gradients['conv_gradients'].mean(axis=(2, 3))


_, N, H, W = hook_f_map.size()

gradcam = torch.matmul(hook_weight, hook_f_map.view(2048, 7*7))

gradcam = gradcam.view(H, W).cpu().detach().numpy()


gradcam = np.maximum(gradcam, 0) #ReLU


gradcam_tensor = torch.from_numpy(gradcam).float().unsqueeze(0).unsqueeze(0)  # Преобразовываем обратно в тензор
interpol = F.interpolate(gradcam_tensor, (224, 224), mode="bilinear").squeeze(0).squeeze(0) # Интерполируем карту на изображение


plt.axis('off')
plt.imshow(display(image))
plt.imshow(interpol, alpha=0.5, cmap='jet')
plt.show()

**Способ 3. Библиотечная реализация.**

В завершение, приведем пример построения GradCAM из библиотеки.

In [ ]:
def schow_library_example(model, prediction, input, original_input, target_layer):

  print('The example started...')
  print('GradCAM is buildinf for prediction: ', prediction)

  targets = [ClassifierOutputTarget(prediction)] # сюда заисываем класс, по которому будем стоить карту
  target_layers = [target_layer]

  with GradCAM(model=model, target_layers=target_layers) as cam:

    grayscale_cam = cam(input_tensor=input, targets=targets)
    cam_image = show_cam_on_image(original_input, grayscale_cam[0, :], use_rgb=True) # строим CAM на rgb изображении

  cam = np.uint8(255*grayscale_cam[0, :])
  cam = cv2.merge([cam, cam, cam])
  images = np.hstack((np.uint8(255*original_input), cam , cam_image))


  return images


original_to_code_example = np.float32(display(image)) / 255

In [ ]:
resnet = models.resnet50(weights=ResNet50_Weights.IMAGENET1K_V1)
resnet.eval()

prediction = resnet(pred).argmax(dim=1)
images = schow_library_example(resnet, prediction, pred, original_to_code_example, resnet.layer4[-1])

Image.fromarray(images)

In [ ]:
#DenseNet
densenet = models.densenet201(weights=DenseNet201_Weights.IMAGENET1K_V1)
densenet.eval()

prediction = densenet(pred).argmax(dim=1)
images = schow_library_example(densenet, prediction, pred, original_to_code_example, densenet.features[-1])

Image.fromarray(images)

В целом, реализаций GradCAM много. С ним можно работать
- на основе [других пользовательских](https://github.com/Aa-Aanegola/Grad-CAM/tree/master) реализаций в pyTorch
- на основе [других пользовательских](https://github.com/ismailuddin/gradcam-tensorflow-2) реализаций в Tensorflow
- на основе других open-source решений, например [captum](https://captum.ai/)

Совет как выбирать — смотрите на наиболее удобную для вас.